# LeetCode 3677 — Count Binary Palindromic Numbers bằng Python

- Link: https://leetcode.com/problems/count-binary-palindromic-numbers/
- Độ khó: **Hard**
- Chủ đề: Bit, **palindrome xác định bởi nửa đầu**, đếm theo độ dài, so sánh với cận trên (kỹ thuật kiểu digit DP), `bit_length`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countBinaryPalindromes(self, n: int) -> int:
```

## 1. Mục tiêu học

1. Một palindrome độ dài `L` được xác định hoàn toàn bởi `h = ⌈L/2⌉` bit đầu (bit đầu tiên phải là 1).
2. Đếm palindrome độ dài `L`: `2^{h−1}`.
3. Với độ dài bằng độ dài của `n`: palindrome `≤ n` ⇔ nửa đầu `< nửa đầu của n`, hoặc **bằng** và palindrome dựng từ nó `≤ n`.
4. Ghép thành công thức `O(log n)`; xử lý riêng `0`.
5. Kiểm chứng bằng vét cạn và bằng sinh palindrome.

## 2. Đề bài & ví dụ

Đếm số `k ∈ [0, n]` có biểu diễn nhị phân (không có số 0 đứng đầu) là palindrome. Quy ước `0 → "0"` là palindrome.

**Ví dụ 1:** `n = 9` → `6` (`0, 1, 3, 5, 7, 9`).

**Ví dụ 2:** `n = 0` → `1`.

**Ràng buộc:** `0 ≤ n ≤ 10^15` (khoảng 50 bit).

In [1]:
import random, time

examples = [(9, 6), (0, 1)]
for n, exp in examples:
    print(n, "-> kỳ vọng", exp)

9 -> kỳ vọng 6
0 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Dựng palindrome từ nửa đầu

Với nửa đầu `p` (số `h` bit) và tổng độ dài `L`: nếu `L` chẵn, nối `p` với `p` đảo ngược; nếu lẻ, bỏ bit cuối của `p` trước khi đảo.

In [2]:
def make_pal(p, L):
    h = (L + 1) // 2
    s = bin(p)[2:]
    assert len(s) == h
    return int(s + s[::-1][L % 2:], 2)

for p, L in [(2, 3), (3, 3), (2, 4), (5, 5)]:
    v = make_pal(p, L); print(f"nửa đầu {bin(p)[2:]}, L={L} -> {bin(v)[2:]} = {v}")

nửa đầu 10, L=3 -> 101 = 5
nửa đầu 11, L=3 -> 111 = 7
nửa đầu 10, L=4 -> 1001 = 9
nửa đầu 101, L=5 -> 10101 = 21


### 3.2. Đơn điệu theo nửa đầu

Với `L` cố định, hai palindrome khác nửa đầu `p₁ < p₂` thì khác nhau đầu tiên trong nửa đầu ⇒ `pal(p₁) < pal(p₂)`. Vậy đếm palindrome `≤ n` độ dài `L` = đếm nửa đầu `p` thoả `pal(p) ≤ n`, là một **khoảng** `[2^{h−1}, p*]`.

### 3.3. `bit_length` và dịch bit

`n.bit_length()` = số bit `L` của `n`; `n >> (L − h)` = `h` bit đầu của `n`.

In [3]:
n = 0b101101
print("L =", n.bit_length(), "| 3 bit đầu =", bin(n >> (n.bit_length() - 3)))

L = 6 | 3 bit đầu = 0b101


### 3.4. Bảng palindrome nhị phân theo độ dài

Liệt kê để thấy quy luật `2^{⌈ℓ/2⌉−1}`: độ dài 1 và 2 có 1 palindrome, độ dài 3 và 4 có 2, độ dài 5 và 6 có 4, … — mỗi lần tăng hai đơn vị độ dài thì nhân đôi (thêm một bit tự do ở nửa đầu).

In [4]:
from collections import defaultdict
by_len = defaultdict(list)
for x in range(1, 1 << 7):
    b = bin(x)[2:]
    if b == b[::-1]: by_len[len(b)].append(b)
for l in sorted(by_len):
    print(f"độ dài {l}: {len(by_len[l]):>2} palindrome, công thức 2^(ceil({l}/2)-1) = {1 << ((l + 1) // 2 - 1)} | ví dụ {by_len[l][:4]}")

độ dài 1:  1 palindrome, công thức 2^(ceil(1/2)-1) = 1 | ví dụ ['1']
độ dài 2:  1 palindrome, công thức 2^(ceil(2/2)-1) = 1 | ví dụ ['11']
độ dài 3:  2 palindrome, công thức 2^(ceil(3/2)-1) = 2 | ví dụ ['101', '111']
độ dài 4:  2 palindrome, công thức 2^(ceil(4/2)-1) = 2 | ví dụ ['1001', '1111']
độ dài 5:  4 palindrome, công thức 2^(ceil(5/2)-1) = 4 | ví dụ ['10001', '10101', '11011', '11111']
độ dài 6:  4 palindrome, công thức 2^(ceil(6/2)-1) = 4 | ví dụ ['100001', '101101', '110011', '111111']
độ dài 7:  8 palindrome, công thức 2^(ceil(7/2)-1) = 8 | ví dụ ['1000001', '1001001', '1010101', '1011101']


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đặt `L = bit_length(n)`:

- Mọi palindrome độ dài `ℓ < L` đều `< n` ⇒ cộng `Σ_{ℓ=1}^{L−1} 2^{⌈ℓ/2⌉−1}`.
- Độ dài `L`: `top = n >> (L − h)`; nửa đầu `p ∈ [2^{h−1}, top − 1]` cho palindrome `< n` (đủ `top − 2^{h−1}` số); thêm 1 nếu `pal(top) ≤ n`.
- Cộng 1 cho số `0`.

Ví dụ `n = 9 = 1001₂`, `L = 4`: độ dài 1: `1`; 2: `11`; 3: `101`, `111` ⇒ 4. Độ dài 4: `h = 2`, `top = 10₂ = 2`, `p ∈ [2, 1]` rỗng; `pal(10₂) = 1001₂ = 9 ≤ 9` ⇒ +1. Thêm `0` ⇒ `6`. ✔

## 5. Cách 1 — Brute Force

In [5]:
def brute(n):
    return sum(1 for k in range(n + 1) if bin(k)[2:] == bin(k)[2:][::-1])

for n, exp in examples:
    print(brute(n), "kỳ vọng", exp)

6 kỳ vọng 6
1 kỳ vọng 1


**Nút thắt:** `O(n log n)` — `10^15` là bất khả thi.

## 6. Ý tưởng thuật toán / bước đột phá

Đếm theo độ dài + so nửa đầu (mục 4): `O(log n)` phép.

Công thức tổng các độ dài nhỏ hơn cũng có dạng đóng: với `L − 1` độ dài, mỗi cặp độ dài `(2t−1, 2t)` góp `2·2^{t−1} = 2^t` ⇒ tổng là một cấp số nhân, nhưng vòng lặp ≤ 50 bước đã đủ nhanh và ít lỗi hơn.

Một cách nhìn tương đương là **digit DP** trên nửa đầu với cờ `tight`, nhưng vì palindrome đơn điệu theo nửa đầu nên chỉ cần **một** phép so cuối cùng.

## 7. Các cách tiếp theo

- **Cách 2:** sinh mọi palindrome nhị phân `≤ n` từ nửa đầu — `O(√n)` palindrome, chỉ cho `n` vừa.
- **Cách 3:** công thức đếm theo độ dài (lời giải nộp).
- **Cách 4:** tìm nhị phân trên nửa đầu: với `L` cố định, tìm `p*` lớn nhất có `pal(p*) ≤ n` bằng tìm nhị phân (thay cho lập luận "top hoặc top − 1").

### 7.1. Cách 2 — Sinh palindrome

In [6]:
def generate(n):
    if n == 0: return 1
    cnt = 1                                            # số 0
    for L in range(1, n.bit_length() + 1):
        h = (L + 1) // 2
        for p in range(1 << (h - 1), 1 << h):
            if make_pal(p, L) <= n: cnt += 1
            else: break                                 # đơn điệu: dừng sớm
    return cnt

for n, exp in examples:
    print(generate(n), "kỳ vọng", exp)

6 kỳ vọng 6
1 kỳ vọng 1


### 7.2. Cách 3 — Công thức (lời giải chính)

In [7]:
def formula(n):
    if n == 0: return 1
    L = n.bit_length(); cnt = 1                       # số 0
    for l in range(1, L):
        cnt += 1 << ((l + 1) // 2 - 1)
    h = (L + 1) // 2
    top = n >> (L - h)
    cnt += top - (1 << (h - 1))                        # nửa đầu nhỏ hơn top
    if make_pal(top, L) <= n: cnt += 1
    return cnt

for n, exp in examples:
    print(formula(n), "kỳ vọng", exp)

6 kỳ vọng 6
1 kỳ vọng 1


### 7.3. Cách 4 — Tìm nhị phân trên nửa đầu

In [8]:
def binary_search_half(n):
    if n == 0: return 1
    L = n.bit_length(); cnt = 1
    for l in range(1, L):
        cnt += 1 << ((l + 1) // 2 - 1)
    h = (L + 1) // 2; lo, hi = (1 << (h - 1)) - 1, (1 << h) - 1   # tìm p lớn nhất có pal(p) <= n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if make_pal(mid, L) <= n: lo = mid
        else: hi = mid - 1
    return cnt + lo - ((1 << (h - 1)) - 1)

for n, exp in examples:
    print(binary_search_half(n), "kỳ vọng", exp)

6 kỳ vọng 6
1 kỳ vọng 1


### 7.4. Cách 5 — Digit DP trên nửa đầu

Viết rõ "kỹ thuật digit DP" cho nửa đầu `h` bit của độ dài `L`: duyệt bit từ cao xuống với cờ `tight` (đang bằng tiền tố của `top`). Khi `tight` tắt, các bit còn lại tự do ⇒ cộng `2^{số bit còn lại}`. Khi đi hết mà vẫn `tight` (nửa đầu = `top`), so palindrome với `n`. Kết quả trùng với `top − 2^{h−1} + [pal(top) ≤ n]` — Cách 3 chính là phiên bản "nén" của DP này.

In [9]:
def digit_dp_half(n):
    if n == 0: return 1
    L = n.bit_length(); cnt = 1
    for l in range(1, L): cnt += 1 << ((l + 1) // 2 - 1)
    h = (L + 1) // 2; top = n >> (L - h); bits = bin(top)[2:]
    # bit đầu bắt buộc là 1 = bit đầu của top
    for i in range(1, h):
        if bits[i] == "1":                       # đặt 0 ở đây -> nhỏ hơn top, các bit sau tự do
            cnt += 1 << (h - 1 - i)
    cnt += make_pal(top, L) <= n                 # vẫn tight tới cuối
    return cnt

print([digit_dp_half(n) for n, _ in examples], "| khớp Cách 3 với n < 5000:", all(digit_dp_half(n) == formula(n) for n in range(5000)))

[6, 1] | khớp Cách 3 với n < 5000: True


## 8. TRACE từng bước

In [10]:
def trace(n):
    L = n.bit_length(); print(f"n = {n} = {bin(n)[2:]}, L = {L}")
    cnt = 1; print("  số 0: +1")
    for l in range(1, L):
        c = 1 << ((l + 1) // 2 - 1); cnt += c; print(f"  độ dài {l}: +{c}")
    h = (L + 1) // 2; top = n >> (L - h)
    print(f"  độ dài {L}: h = {h}, top = {bin(top)[2:]}; nửa đầu nhỏ hơn: +{top - (1 << (h - 1))}")
    cnt += top - (1 << (h - 1))
    p = make_pal(top, L)
    print(f"  pal(top) = {bin(p)[2:]} = {p} {'<=' if p <= n else '>'} n -> +{int(p <= n)}")
    cnt += p <= n
    print("  tổng:", cnt, "| brute:", brute(n))

trace(9)
trace(44)

n = 9 = 1001, L = 4
  số 0: +1
  độ dài 1: +1
  độ dài 2: +1
  độ dài 3: +2
  độ dài 4: h = 2, top = 10; nửa đầu nhỏ hơn: +0
  pal(top) = 1001 = 9 <= n -> +1
  tổng: 6 | brute: 6
n = 44 = 101100, L = 6
  số 0: +1
  độ dài 1: +1
  độ dài 2: +1
  độ dài 3: +2
  độ dài 4: +2
  độ dài 5: +4
  độ dài 6: h = 3, top = 101; nửa đầu nhỏ hơn: +1
  pal(top) = 101101 = 45 > n -> +0
  tổng: 12 | brute: 12


## 9. Code nộp LeetCode

In [11]:
class Solution:
    def countBinaryPalindromes(self, n: int) -> int:
        if n == 0:
            return 1
        L = n.bit_length()
        cnt = 1                                   # số 0
        for l in range(1, L):
            cnt += 1 << ((l + 1) // 2 - 1)
        h = (L + 1) // 2
        top = n >> (L - h)
        cnt += top - (1 << (h - 1))
        s = bin(top)[2:]
        pal = int(s + s[::-1][L % 2:], 2)
        if pal <= n:
            cnt += 1
        return cnt

sol = Solution()
for n, exp in examples:
    got = sol.countBinaryPalindromes(n)
    print(n, "->", got, "✔" if got == exp else "✘")

9 -> 6 ✔
0 -> 1 ✔


## 10. Vì sao thuật toán đúng

1. **Song ánh nửa đầu:** palindrome độ dài `ℓ` (bit đầu = 1) ↔ chuỗi `h = ⌈ℓ/2⌉` bit bắt đầu bằng 1 ⇒ có `2^{h−1}` palindrome độ dài `ℓ`.
2. **Độ dài nhỏ hơn:** mọi số có ít bit hơn `n` đều nhỏ hơn `n`.
3. **Độ dài bằng `L`:** palindrome `P` với nửa đầu `p`; `h` bit đầu của `n` là `top`. Nếu `p < top` thì `P < n` (khác nhau trong `h` bit đầu); nếu `p > top` thì `P > n`; nếu `p = top` thì cần so trực tiếp. Số `p` trong `[2^{h−1}, top)` là `top − 2^{h−1}` (lưu ý `top ≥ 2^{h−1}` vì bit đầu của `n` là 1).
4. **Số 0** được đếm riêng (biểu diễn `"0"`).
5. **Không trùng, không sót:** mỗi số `k ∈ [1, n]` có đúng một độ dài `ℓ = bit_length(k)`; với `ℓ < L` nó được đếm ở bước 2, với `ℓ = L` ở bước 3; số `0` ở bước 4. Các nhóm rời nhau và phủ hết `[0, n]`.

**Ví dụ kiểm tra biên:** `n = 2^e` (một bit 1 rồi toàn 0). Khi đó `top = 2^{h−1}` ⇒ không có nửa đầu nào nhỏ hơn, và `pal(top) = 100…001₂ > n` ⇒ không cộng gì ở độ dài `L` (trừ `e = 0`, `n = 1`: `pal = 1 = n`). Vậy `f(2^e) = f(2^e − 1)` với `e ≥ 1` — khớp trực giác: `2^e` không phải palindrome.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n log n)` | `O(1)` |
| 2. Sinh palindrome | `O(√n · log n)` | `O(1)` |
| 3. Công thức | `O(log n)` | `O(log n)` (chuỗi bit) |
| 4. Tìm nhị phân nửa đầu | `O(log² n)` | `O(log n)` |

## 12. Bộ test

In [12]:
METHODS = [brute, generate, formula, binary_search_half, digit_dp_half, lambda n: Solution().countBinaryPalindromes(n)]
for n, exp in examples:
    assert all(f(n) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 mọi n = 0..20000
pref = 0
for n in range(0, 20001):
    if bin(n)[2:] == bin(n)[2:][::-1]: pref += 1
    assert formula(n) == pref == Solution().countBinaryPalindromes(n), n
    if n < 3000: assert generate(n) == binary_search_half(n) == pref
print("n = 0..20000: OK")

n = 0..20000: OK


In [14]:
# 12.3 n lớn ngẫu nhiên và các lũy thừa 2
rng = random.Random(3677)
for _ in range(500):
    n = rng.randint(0, 10**15)
    assert formula(n) == binary_search_half(n) == Solution().countBinaryPalindromes(n)
for e in range(1, 50):
    for n in (2**e - 1, 2**e, 2**e + 1):
        assert formula(n) == binary_search_half(n)
print("n lớn: OK | n = 10^15 ->", Solution().countBinaryPalindromes(10**15))

n lớn: OK | n = 10^15 -> 63356754


In [15]:
# 12.4 thời gian
t0 = time.time()
for _ in range(10**4): Solution().countBinaryPalindromes(10**15)
print("10^4 lần với n = 10^15:", round(time.time() - t0, 3), "s")

10^4 lần với n = 10^15: 0.046 s


**Kiểm tra tính chất biên `f(2^e) = f(2^e − 1)`** (mục 10) và quan sát tốc độ tăng: số palindrome nhị phân `≤ n` xấp xỉ `c·√n` — vì mỗi palindrome độ dài `L` ứng với một nửa đầu khoảng `L/2` bit.

In [16]:
for e in range(1, 12):
    assert formula(2**e) == formula(2**e - 1)
import math
for n in [10**3, 10**6, 10**9, 10**12, 10**15]:
    c = formula(n); print(f"n = {n:>16}: {c:>9} palindrome, tỉ lệ với sqrt(n) = {c / math.isqrt(n):.3f}")

n =             1000:        62 palindrome, tỉ lệ với sqrt(n) = 2.000
n =          1000000:      2000 palindrome, tỉ lệ với sqrt(n) = 2.000
n =       1000000000:     63284 palindrome, tỉ lệ với sqrt(n) = 2.001
n =    1000000000000:   2002249 palindrome, tỉ lệ với sqrt(n) = 2.002
n = 1000000000000000:  63356754 palindrome, tỉ lệ với sqrt(n) = 2.004


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên đếm số 0

In [17]:
def wrong_no_zero(n):
    return formula(n) - 1 if n > 0 else 0                              # SAI: bỏ số 0

print("sai:", wrong_no_zero(9), "| đúng:", brute(9))

sai: 5 | đúng: 6


### Lỗi 2 — Đếm `2^h` palindrome mỗi độ dài (quên bit đầu bắt buộc là 1)

In [18]:
def wrong_count_per_length(n):
    L = n.bit_length(); cnt = 1
    for l in range(1, L): cnt += 1 << ((l + 1) // 2)                     # SAI
    h = (L + 1) // 2; top = n >> (L - h)
    return cnt + top - (1 << (h - 1)) + (make_pal(top, L) <= n)

print("sai:", wrong_count_per_length(9), "| đúng:", brute(9))

sai: 10 | đúng: 6


### Lỗi 3 — Luôn cộng palindrome của `top` (không so với `n`)

In [19]:
def wrong_always_top(n):
    L = n.bit_length(); cnt = 1
    for l in range(1, L): cnt += 1 << ((l + 1) // 2 - 1)
    h = (L + 1) // 2; top = n >> (L - h)
    return cnt + top - (1 << (h - 1)) + 1                                 # SAI

print("sai:", wrong_always_top(8), "| đúng:", brute(8))

sai: 6 | đúng: 5


### Lỗi 4 — Dựng palindrome độ dài lẻ mà không bỏ bit giữa (lặp bit giữa)

In [20]:
def wrong_mirror(p, L):
    s = bin(p)[2:]; return int(s + s[::-1], 2)                            # SAI khi L lẻ

print("p = 10₂, L = 3: sai", bin(wrong_mirror(2, 3))[2:], "| đúng", bin(make_pal(2, 3))[2:])

p = 10₂, L = 3: sai 1001 | đúng 101


### Lỗi 5 — Dùng `⌊ℓ/2⌋` thay cho `⌈ℓ/2⌉` làm số bit tự do

Với độ dài **lẻ**, bit giữa cũng là bit tự do (nó tự đối xứng với chính nó). Nếu dùng `⌊ℓ/2⌋`, ta bỏ mất bit giữa ⇒ đếm thiếu một nửa số palindrome độ dài lẻ (ví dụ độ dài 3 chỉ đếm được 1 thay vì 2: thiếu `111` hoặc `101`). Ô dưới cài bản sai (chỉ phần đếm theo độ dài nhỏ hơn) để thấy số lượng lệch ngay từ `n = 9`.

In [21]:
def wrong_half_len(n):
    if n == 0: return 1
    L = n.bit_length(); cnt = 1
    for l in range(1, L): cnt += 1 << (l // 2 - 1) if l >= 2 else 1           # SAI: dùng floor(l/2)
    return cnt

print("sai:", wrong_half_len(9), "| đúng:", brute(9))

sai: 4 | đúng: 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm palindrome ≤ n" (cơ số bất kỳ) ⇒ đếm theo độ dài + so nửa đầu với nửa đầu của `n`.
- Palindrome đơn điệu theo nửa đầu ⇒ chỉ cần một phép so ở biên.
- Cẩn thận ca suy biến: `0`, `n` là lũy thừa của 2, độ dài lẻ/chẵn.
- Số palindrome `≤ n` chỉ cỡ `√n` ⇒ khi `n ≤ 10^{10}` có thể **sinh** trực tiếp; với `10^{15}` vẫn sinh được (≈ `10^7`) nhưng chậm trong Python — công thức `O(log n)` là an toàn nhất.
- Dựng palindrome bằng chuỗi `bin(p)[2:]` dễ đọc; muốn nhanh hơn có thể đảo bit bằng vòng lặp số học.

## 15. Biến thể của bài

1. **Palindrome thập phân `≤ n`**: cùng ý tưởng, mỗi độ dài có `9 · 10^{h−1}` palindrome.
2. **Đếm trên đoạn `[a, b]`**: `f(b) − f(a − 1)`.
3. **Palindrome nhị phân thứ `k`**: chọn độ dài theo tổng tích luỹ, rồi nửa đầu = `2^{h−1} + (chỉ số trong độ dài)`.
4. **Số vừa palindrome nhị phân vừa thập phân** (LC Project-Euler-style): sinh palindrome một cơ số, kiểm tra cơ số kia.

In [22]:
# Biến thể 1 + 3
def count_dec_pal(n):
    if n == 0: return 1
    s = str(n); L = len(s); cnt = 1
    for l in range(1, L): cnt += 9 * 10 ** ((l + 1) // 2 - 1)
    h = (L + 1) // 2; top = int(s[:h])
    cnt += top - 10 ** (h - 1)
    t = str(top); cnt += int(t + t[::-1][L % 2:]) <= n
    return cnt

def kth_bin_pal(k):                                      # k = 1 -> 0
    if k == 1: return 0
    k -= 1; L = 1
    while True:
        c = 1 << ((L + 1) // 2 - 1)
        if k <= c: return make_pal((1 << ((L + 1) // 2 - 1)) + k - 1, L)
        k -= c; L += 1

assert all(count_dec_pal(n) == sum(1 for x in range(n + 1) if str(x) == str(x)[::-1]) for n in range(0, 3000))
pals = [x for x in range(3000) if bin(x)[2:] == bin(x)[2:][::-1]]
assert [kth_bin_pal(k) for k in range(1, len(pals) + 1)] == pals
print("Biến thể 1, 3 OK; 10 palindrome nhị phân đầu:", [kth_bin_pal(k) for k in range(1, 11)])

Biến thể 1, 3 OK; 10 palindrome nhị phân đầu: [0, 1, 3, 5, 7, 9, 15, 17, 21, 27]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Kiểm tra từng số | `O(n log n)` | Kiểm chứng |
| 2. Sinh palindrome | Từ nửa đầu | `O(√n)` | `n` vừa, cần liệt kê |
| 3. Công thức | Độ dài + so biên | `O(log n)` | **Lời giải chuẩn** |
| 4. Nhị phân nửa đầu | Đơn điệu | `O(log² n)` | Khi khó lập luận biên |
| 5. Digit DP nửa đầu | Cờ `tight` | `O(log n)` | Mẫu tổng quát cho ràng buộc phức tạp hơn |

Cả năm cách đều cho cùng kết quả trên mọi `n ≤ 20000` (mục 12); Cách 3 ngắn nhất, Cách 5 dễ mở rộng nhất (ví dụ thêm điều kiện "số bit 1 chẵn").

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countBinaryPalindromes(7)`.

**Bài 2 (tính tay).** `countBinaryPalindromes(31)`.

**Bài 3 (điền TODO).** Viết `count_in_range(a, b)`.

**Bài 4 (tự viết lại).** Viết lời giải cho **cơ số `B` bất kỳ** (`2 ≤ B ≤ 36`).

**Bài 5 (mini challenge).** Tìm palindrome nhị phân **nhỏ nhất `> n`**.

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_in_range(a, b):
    # TODO
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def next_bin_pal(n):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `0, 1, 11, 101, 111`. (2) `0, 1, 3, 5, 7`. (3) `5`.

**Bài 2.** (1) `L = 5`, `31 = 11111₂`. (2) Độ dài 1..4: `1 + 1 + 2 + 2 = 6`; độ dài 5: `top = 111₂`, nửa đầu `100, 101, 110` + `pal(111) = 11111 ≤ 31` ⇒ 4. (3) `1 + 6 + 4 = 11`.

**Bài 3.** (1) `f(b) − f(a − 1)`. (2) `a = 0` ⇒ chỉ `f(b)`. (3) Kiểm chứng.

**Bài 4.** (1) Đổi `bit_length` thành số chữ số cơ số `B`. (2) Mỗi độ dài có `(B−1)·B^{h−1}`. (3) Dựng palindrome bằng danh sách chữ số.

**Bài 5.** (1) Đếm `c = f(n)`. (2) Palindrome thứ `c + 1` (dùng `kth_bin_pal`). (3) Kiểm tra bằng vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().countBinaryPalindromes(7) == 5 == brute(7)
assert Solution().countBinaryPalindromes(31) == 11 == brute(31)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3, 4
def count_in_range_sol(a, b):
    return formula(b) - (formula(a - 1) if a > 0 else 0)

def to_digits(x, B):
    d = []
    while x: d.append(x % B); x //= B
    return d[::-1] or [0]

def count_base(n, B):
    if n == 0: return 1
    D = to_digits(n, B); L = len(D); cnt = 1
    for l in range(1, L): cnt += (B - 1) * B ** ((l + 1) // 2 - 1)
    h = (L + 1) // 2; top = 0
    for d in D[:h]: top = top * B + d
    cnt += top - B ** (h - 1)
    td = to_digits(top, B); pal = td + td[::-1][L % 2:]
    v = 0
    for d in pal: v = v * B + d
    return cnt + (v <= n)

rng = random.Random(4)
for _ in range(300):
    a = rng.randint(0, 5000); b = rng.randint(a, 6000)
    assert count_in_range_sol(a, b) == sum(1 for x in range(a, b + 1) if bin(x)[2:] == bin(x)[2:][::-1])
for B in (2, 3, 7, 10, 16):
    for n in range(0, 1500):
        assert count_base(n, B) == sum(1 for x in range(n + 1) if to_digits(x, B) == to_digits(x, B)[::-1]), (n, B)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [27]:
# Bài 5
def next_bin_pal_sol(n):
    return kth_bin_pal(formula(n) + 1)

for n in range(0, 3000):
    x = n + 1
    while bin(x)[2:] != bin(x)[2:][::-1]: x += 1
    assert next_bin_pal_sol(n) == x
print("Bài 5 OK; palindrome nhị phân kế tiếp sau 10^15:", next_bin_pal_sol(10**15))

Bài 5 OK; palindrome nhị phân kế tiếp sau 10^15: 1000000047151815


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if n == 0: return 1
L = n.bit_length(); cnt = 1 + Σ_{l<L} 2^(ceil(l/2)-1)
h = ceil(L/2); top = n >> (L - h)
cnt += top - 2^(h-1)
cnt += pal(top, L) <= n
```

**Checklist:**

- [ ] Có đếm số 0?
- [ ] Mỗi độ dài `2^{h−1}` (bit đầu là 1)?
- [ ] So `pal(top)` với `n`?
- [ ] Độ dài lẻ bỏ bit giữa khi lật?
- [ ] Đã so với vét cạn `n ≤ 20000`?
- [ ] `n` là lũy thừa của 2 cho kết quả bằng `n − 1`?